In [0]:
from pyspark.sql.functions import col

df_silver = spark.read.table("workspace.silver.silver_customers")

total_rows = df_silver.count()

null_customer_id = df_silver.filter(col("customer_id").isNull()).count()
null_subscription_date = df_silver.filter(col("subscription_date").isNull()).count()
null_email = df_silver.filter(col("email").isNull()).count()

duplicate_customer_id = (
    df_silver
    .groupBy("customer_id")
    .count()
    .filter(col("count") > 1)
    .count()
)

dq_results = [
    ("silver", total_rows, "customer_id_not_null", null_customer_id, "PASS" if null_customer_id == 0 else "FAIL"),
    ("silver", total_rows, "subscription_date_not_null", null_subscription_date, "PASS" if null_subscription_date == 0 else "FAIL"),
    ("silver", total_rows, "email_not_null", null_email, "PASS" if null_email == 0 else "FAIL"),
    ("silver", total_rows, "customer_id_unique", duplicate_customer_id, "PASS" if duplicate_customer_id == 0 else "FAIL")
]

df_dq_summary = spark.createDataFrame(
    dq_results,
    ["layer_name", "total_rows_checked", "check_name", "failed_records", "status"]
)

display(df_dq_summary)


In [0]:
df_dq_summary.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.gold_data_quality_summary")
